# Lab Assignment 2 - Part B: k-Nearest Neighbor Classification
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement the k-Nearest Neighbor (k-NN) classifier and evaluate it on two datasets:
- **Lenses Dataset**: A small dataset for contact lens prescription
- **Credit Approval (CA) Dataset**: Credit card application data with binary labels (+/-)

### Your Tasks
1. **Preprocess the data**: Handle missing values and normalize features
2. **Implement k-NN** with L2 distance
3. **Evaluate** on both datasets for different values of k
4. **Discuss** your results

### Datasets
The data files are located in the `credit 2017/` folder:
- `lenses.training`, `lenses.testing`
- `crx.data.training`, `crx.data.testing`
- `crx.names` (describes the features)


## Setup


In [2]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [3]:
# Data paths
DATA_PATH = "credit 2017/"

# Load Lenses data
def load_lenses_data():
    """Load the lenses dataset."""
    train_data = np.loadtxt(DATA_PATH + "lenses.training", delimiter=',')
    test_data = np.loadtxt(DATA_PATH + "lenses.testing", delimiter=',')
    
    # First column is ID, last column is label
    X_train = train_data[:, 1:-1]
    y_train = train_data[:, -1]
    X_test = test_data[:, 1:-1]
    y_test = test_data[:, -1]
    
    return X_train, y_train, X_test, y_test

# Load Credit Approval data
def load_credit_data():
    """
    Load the Credit Approval dataset.
    Note: This dataset contains missing values (?) and mixed types.
    You will need to preprocess it.
    """
    # TODO: Implement data loading
    # The data is comma-separated
    # Missing values are marked with '?'
    # Last column is the label ('+' or '-')
    pass

# Test loading lenses data
X_train_lenses, y_train_lenses, X_test_lenses, y_test_lenses = load_lenses_data()
print(f"Lenses - Train: {X_train_lenses.shape}, Test: {X_test_lenses.shape}")


Lenses - Train: (18, 3), Test: (6, 3)


## Task 1: Data Preprocessing
For the Credit Approval dataset, you need to:
1. **Handle missing values** (marked with '?'):
   - Categorical features: replace with mode/median
   - Numerical features: replace with label-conditioned mean
2. **Normalize features** using z-scaling:
   $$z_i^{(m)} = \frac{x_i^{(m)} - \mu_i}{\sigma_i}$$

Document exactly how you handle each feature!


In [ ]:
def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.
    
    Steps:
    1. Load the data
    2. Handle missing values
    3. Encode categorical variables
    4. Normalize numerical features
    
    Returns:
    --------
    X_train, y_train, X_test, y_test : numpy arrays
    """
    # TODO: Implement preprocessing
    # Hint: Read crx.names to understand the features
    # Feature types (from crx.names):
    # A1: categorical (b, a)
    # A2: continuous
    # A3: continuous
    # A4: categorical (u, y, l, t)
    # A5: categorical (g, p, gg)
    # A6: categorical (c, d, cc, i, j, k, m, r, q, w, x, e, aa, ff)
    # A7: categorical (v, h, bb, j, n, z, dd, ff, o)
    # A8: continuous
    # A9: categorical (t, f)
    # A10: categorical (t, f)
    # A11: continuous
    # A12: categorical (t, f)
    # A13: categorical (g, p, s)
    # A14: continuous
    # A15: continuous
    # A16: label (+, -)
    
    pass


def z_normalize(X_train, X_test, feature_indices):
    """
    Apply z-score normalization to specified features.
    
    Parameters:
    -----------
    X_train, X_test : numpy arrays
    feature_indices : list of indices for numerical features
    
    Returns:
    --------
    X_train_normalized, X_test_normalized : numpy arrays
    """
    # TODO: Implement z-normalization
    pass


In [4]:
def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.
    
    Steps:
    1. Load the data
    2. Handle missing values
    3. Encode categorical variables
    4. Normalize numerical features
    """
    
    # Feature indices (0-based)
    categorical_idx = [0, 3, 4, 5, 6, 8, 9, 11, 12]
    numerical_idx = [1, 2, 7, 10, 13, 14]
    label_idx = 15

    # --------------------
    # 1. Load data
    # --------------------
    def load_file(path):
        data = []
        with open(path, "r") as f:
            for line in f:
                data.append(line.strip().split(","))
        return np.array(data, dtype=object)

    train_raw = load_file(train_file)
    test_raw = load_file(test_file)

    y_train = train_raw[:, label_idx]
    y_test = test_raw[:, label_idx]

    X_train = train_raw[:, :label_idx]
    X_test = test_raw[:, :label_idx]

    # --------------------
    # 2. Handle missing values
    # --------------------

    # --- Categorical: replace '?' with mode ---
    for idx in categorical_idx:
        values = [v for v in X_train[:, idx] if v != "?"]
        mode = Counter(values).most_common(1)[0][0]

        X_train[X_train[:, idx] == "?", idx] = mode
        X_test[X_test[:, idx] == "?", idx] = mode

    # --- Numerical: label-conditioned mean ---
    for idx in numerical_idx:
        for label in ["+", "-"]:
            mask = (y_train == label) & (X_train[:, idx] != "?")
            mean_val = np.mean(X_train[mask, idx].astype(float))

            cond = (y_train == label) & (X_train[:, idx] == "?")
            X_train[cond, idx] = mean_val

            cond_test = (y_test == label) & (X_test[:, idx] == "?")
            X_test[cond_test, idx] = mean_val

        X_train[:, idx] = X_train[:, idx].astype(float)
        X_test[:, idx] = X_test[:, idx].astype(float)

    # --------------------
    # 3. Encode categorical variables
    # --------------------
    category_maps = {}

    for idx in categorical_idx:
        categories = np.unique(X_train[:, idx])
        mapping = {cat: i for i, cat in enumerate(categories)}
        category_maps[idx] = mapping

        X_train[:, idx] = [mapping[v] for v in X_train[:, idx]]
        X_test[:, idx] = [mapping.get(v, -1) for v in X_test[:, idx]]

    X_train = X_train.astype(float)
    X_test = X_test.astype(float)

    # --------------------
    # 4. Normalize numerical features
    # --------------------
    X_train, X_test = z_normalize(X_train, X_test, numerical_idx)

    return X_train, y_train, X_test, y_test

In [5]:
def z_normalize(X_train, X_test, feature_indices):
    """
    Apply z-score normalization to specified features.
    """
    X_train_norm = X_train.copy()
    X_test_norm = X_test.copy()

    for idx in feature_indices:
        mu = np.mean(X_train[:, idx])
        sigma = np.std(X_train[:, idx])

        if sigma == 0:
            continue

        X_train_norm[:, idx] = (X_train[:, idx] - mu) / sigma
        X_test_norm[:, idx] = (X_test[:, idx] - mu) / sigma

    return X_train_norm, X_test_norm

## Task 2: Implement k-NN Classifier
Implement k-NN with L2 (Euclidean) distance:
$$\mathcal{D}_{L2}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

For **categorical attributes**, use:
- Distance = 1 if values are different
- Distance = 0 if values are the same


In [ ]:
def l2_distance(a, b):
    """
    Compute L2 (Euclidean) distance between two vectors.
    
    Parameters:
    -----------
    a, b : numpy arrays of same shape
    
    Returns:
    --------
    distance : float
    """
    # TODO: Implement L2 distance
    pass


def knn_predict(X_train, y_train, X_test, k):
    """
    Predict labels for test data using k-NN.
    
    Parameters:
    -----------
    X_train : numpy array of shape (n_train, n_features)
    y_train : numpy array of shape (n_train,)
    X_test : numpy array of shape (n_test, n_features)
    k : int, number of neighbors
    
    Returns:
    --------
    predictions : numpy array of shape (n_test,)
    """
    # TODO: Implement k-NN prediction
    # For each test sample:
    #   1. Compute distance to all training samples
    #   2. Find k nearest neighbors
    #   3. Predict using majority voting
    pass


def compute_accuracy(y_true, y_pred):
    """
    Compute classification accuracy.
    
    Returns:
    --------
    accuracy : float (between 0 and 1)
    """
    # TODO: Implement accuracy computation
    pass


In [6]:
def l2_distance(a, b):
    """
    Compute L2 (Euclidean) distance between two vectors.
    """
    return np.sqrt(np.sum((a - b) ** 2))

In [7]:
def knn_predict(X_train, y_train, X_test, k):
    """
    Predict labels for test data using k-NN.
    """
    predictions = []

    for x_test in X_test:
        distances = []

        # 1. Compute distance to all training samples
        for x_train, y in zip(X_train, y_train):
            d = l2_distance(x_test, x_train)
            distances.append((d, y))

        # 2. Find k nearest neighbors
        distances.sort(key=lambda x: x[0])
        k_labels = [label for _, label in distances[:k]]

        # 3. Majority voting
        pred = Counter(k_labels).most_common(1)[0][0]
        predictions.append(pred)

    return np.array(predictions)

In [8]:
def compute_accuracy(y_true, y_pred):
    """
    Compute classification accuracy.
    """
    return np.mean(y_true == y_pred)

## Task 3: Evaluate on Lenses Dataset
Test your k-NN implementation on the Lenses dataset for different values of k.


In [ ]:
# TODO: Evaluate k-NN on Lenses dataset
# Try different values of k (e.g., 1, 3, 5, 7)

# k_values = [1, 3, 5, 7]
# lenses_results = []
# 
# for k in k_values:
#     predictions = knn_predict(X_train_lenses, y_train_lenses, X_test_lenses, k)
#     accuracy = compute_accuracy(y_test_lenses, predictions)
#     lenses_results.append((k, accuracy))
#     print(f"k={k}: Accuracy = {accuracy:.4f}")


In [9]:
# Evaluate k-NN on Lenses dataset
# Try different values of k

k_values = [1, 3, 5, 7]
lenses_results = []

for k in k_values:
    predictions = knn_predict(
        X_train_lenses,
        y_train_lenses,
        X_test_lenses,
        k
    )
    accuracy = compute_accuracy(y_test_lenses, predictions)
    lenses_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")

k=1: Accuracy = 1.0000
k=3: Accuracy = 1.0000
k=5: Accuracy = 1.0000
k=7: Accuracy = 1.0000


## Task 4: Evaluate on Credit Approval Dataset
First preprocess the data, then evaluate k-NN.


In [ ]:
# TODO: Preprocess Credit Approval data
# X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
#     DATA_PATH + "crx.data.training",
#     DATA_PATH + "crx.data.testing"
# )


In [10]:
# Preprocess Credit Approval data
X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
    DATA_PATH + "crx.data.training",
    DATA_PATH + "crx.data.testing"
)

print("Credit Approval - Train shape:", X_train_credit.shape)
print("Credit Approval - Test shape:", X_test_credit.shape)

Credit Approval - Train shape: (552, 15)
Credit Approval - Test shape: (138, 15)


In [ ]:
# TODO: Evaluate k-NN on Credit Approval dataset
# k_values = [1, 3, 5, 7]
# credit_results = []
# 
# for k in k_values:
#     predictions = knn_predict(X_train_credit, y_train_credit, X_test_credit, k)
#     accuracy = compute_accuracy(y_test_credit, predictions)
#     credit_results.append((k, accuracy))
#     print(f"k={k}: Accuracy = {accuracy:.4f}")


In [11]:
# Evaluate k-NN on Credit Approval dataset
k_values = [1, 3, 5, 7]
credit_results = []

for k in k_values:
    predictions = knn_predict(
        X_train_credit,
        y_train_credit,
        X_test_credit,
        k
    )
    accuracy = compute_accuracy(y_test_credit, predictions)
    credit_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")

k=1: Accuracy = 0.7464
k=3: Accuracy = 0.7826
k=5: Accuracy = 0.8043
k=7: Accuracy = 0.7971


## Summary and Discussion

### Results Table

| Dataset | k=1 | k=3 | k=5 | k=7 |
|---------|-----|-----|-----|-----|
| Lenses | ? | ? | ? | ? |
| Credit Approval | ? | ? | ? | ? |

### Discussion
*Answer these questions:*
1. Which value of k works best for each dataset? Why do you think that is?
2. How did preprocessing affect your results on the Credit Approval dataset?
3. What are the trade-offs of using different values of k?
4. What did you learn from this exercise?


| Dataset             | k = 1  | k = 3  | k = 5  | k = 7  |
| ------------------- | ------ | ------ | ------ | ------ |
| **Lenses**          | 1.0000 | 1.0000 | 1.0000 | 1.0000 |
| **Credit Approval** | 0.7464 | 0.7826 | 0.8043 | 0.7971 |


1. Which value of k works best for each dataset? Why?
Lenses dataset:
All tested values of k perform equally well (100% accuracy). This indicates that the dataset is small, clean, and well-separated, so increasing k does not introduce conflicting neighbors.
Credit Approval dataset:
k = 5 achieves the highest accuracy (0.8043).
Smaller k values (e.g., k = 1) are more sensitive to noise and outliers, while larger k values (k = 7) begin to include neighbors from different classes, slightly reducing performance.

2. How did preprocessing affect results on the Credit Approval dataset?
Preprocessing had a significant impact on performance:
Handling missing values prevented invalid distance calculations
Label-conditioned mean imputation preserved class-specific patterns
Encoding categorical variables allowed mixed-type features to be used in k-NN
Z-normalization ensured no single numerical feature dominated the distance metric
Without these steps, k-NN would perform poorly due to scale imbalance and missing data.

3. What are the trade-offs of using different values of k?
Small k (k = 1):
Low bias, high variance
Highly sensitive to noisy points
Large k (k = 7):
Lower variance, higher bias
May oversmooth decision boundaries
Moderate k (k = 3 or 5):
Best balance between bias and variance
More robust and stable predictions

4. What did you learn from this exercise?
This exercise demonstrates that:
k-NN is highly sensitive to data preprocessing
Feature scaling and missing value handling are crucial for distance-based methods
Dataset complexity strongly affects model behavior
Even simple models can perform competitively with proper preprocessing